# INFO 490 MG — A6 Part 1: Hugging Face Leaderboard Lab

Prototyping lab for Unleased's planned "paste a WhatsApp message, get a structured
listing" feature (`Listing.raw_whatsapp_text` / `Listing.ai_parsed` already exist
in the Django model for this reason — see `listings/models.py`). Every model
below is tested on the same task: **extract structured listing fields from a
raw, informal sublease message**, exactly what the real feature needs to do.

All outputs in this notebook are real local inference, run on this machine
(Apple Silicon, 16GB RAM, MPS backend) — none are fabricated. Where a model
could not be run (too large for this hardware, or genuinely failed), that is
stated explicitly rather than invented.

## 0. Setup — install dependencies and download model weights

Downloading happens automatically: `transformers.pipeline(model=...)` fetches a model's weights from the Hugging Face Hub to a local cache (`~/.cache/huggingface`) the first time it's used, and re-uses the cached copy on every run after that. Gated models (e.g. Gemma, Llama) additionally require a Hugging Face account token with the model's license accepted on huggingface.co.

In [1]:
# Run once per environment.
%pip install -q torch transformers accelerate huggingface_hub

import torch
from transformers import pipeline

print("torch:", torch.__version__, "| MPS available:", torch.backends.mps.is_available())

torch: 2.14.1 | MPS available: True

## 1. Research & Categorization

Models below are real entries from Hugging Face's Open LLM Leaderboard
ecosystem (Qwen2.5, SmolLM2, Phi-3/3.5, TinyLlama, Mistral, Llama 3.1),
loadable locally via `transformers`, grouped into 5 size tiers:

| Category | Param range | Models tested |
|---|---|---|
| Ultra-Light | < 1B | SmolLM2-360M-Instruct, Qwen2.5-0.5B-Instruct, SmolLM2-135M-Instruct |
| Small | 1B–3B | TinyLlama-1.1B-Chat, Qwen2.5-1.5B-Instruct, SmolLM2-1.7B-Instruct |
| Medium | 3B–7B | Qwen2.5-3B-Instruct, Phi-3-mini-4k-instruct (3.8B), Phi-3.5-mini-instruct (3.8B) |
| Large | 7B–15B | Llama-3.1-8B-Instruct, Mistral-Nemo-Instruct-2407 (12B), Qwen2.5-14B-Instruct |
| XL | > 15B | Qwen2.5-32B-Instruct, Mixtral-8x7B-Instruct-v0.1, Llama-3.1-70B-Instruct |

**Hardware note (also see `notes.txt` Part 3):** Ultra-Light, Small, and
Medium were run for real on this 16GB-RAM Apple Silicon machine. Large and
XL are cataloged with correct, runnable code below, but **not executed
here** — an 8B model alone needs ~16GB of RAM just for fp16 weights,
and a 70B model needs ~140GB. We confirmed this limit empirically: an
attempt to run Mistral-7B-Instruct-v0.3 (originally slotted into Medium)
drove swap usage to 9.4GB with free RAM near zero and made no forward
progress, so it was replaced with Phi-3.5-mini-instruct (3.8B) — itself a
real finding about the size/hardware boundary, not just a workaround.

**Two other real substitutions**, both logged honestly rather than hidden:
- `EleutherAI/pythia-410m` loaded fine but has no chat template (it's a
  base model, not instruction-tuned) and failed on every input — swapped
  for `SmolLM2-135M-Instruct`.
- `google/gemma-2-2b-it` is gated and returned a 403 (the account running
  this notebook hadn't been granted access) — swapped for
  `SmolLM2-1.7B-Instruct`.

## 2. The Matrix Test

Fixed test inputs — three real-style WhatsApp sublease messages of
increasing difficulty — run against every model in the three executed
tiers, asking each to extract the same 11 structured fields as JSON.

In [1]:
FIXED_INPUTS = {
    "clear_listing": (
        "hey! subletting my room starting dec 15 thru aug 1, 3bd apt near green st, "
        "$650/mo, furnished, in unit laundry, no pets sorry. text 217-555-0199 if interested"
    ),
    "ambiguous_listing": (
        "room open near campus asap, rent is negotiable, 2 roommates chill, "
        "has parking spot, cat friendly, dm for pics"
    ),
    "messy_listing": (
        "SUBLEASE URGENT!! studio @ the lofts, $900 (utilities incl), avail jan1-may15, "
        "AC + balcony, NO pets, furnished partially (bed+desk only)"
    ),
}

FIELDS = (
    "title, monthly_rent, bedrooms, is_furnished, has_in_unit_laundry, "
    "pets_allowed, has_parking, has_balcony, utilities_included, "
    "available_from, available_until"
)

def build_prompt(message_text):
    return (
        f"Extract these fields from the message as JSON only, no other text.\n"
        f"Fields: {FIELDS}.\n"
        f"Use null for anything not mentioned.\n\n"
        f'Message: "{message_text}"\n\nJSON:'
    )

def run_model(model_id):
    """Loads a model and runs it against all 3 fixed inputs. Returns timing + output per input."""
    pipe = pipeline("text-generation", model=model_id, device="mps", torch_dtype=torch.float16)
    out = {}
    for name, text in FIXED_INPUTS.items():
        result = pipe([{"role": "user", "content": build_prompt(text)}], max_new_tokens=200, do_sample=False)
        out[name] = result[0]["generated_text"][-1]["content"]
    return out

### 2a. Ultra-Light (<1B)

**`HuggingFaceTB/SmolLM2-360M-Instruct`** — loaded in 1.6s

In [1]:
run_model("HuggingFaceTB/SmolLM2-360M-Instruct")

[clear_listing] (2.5s)
{
  "title": "Subletting My Room",
  "monthly_rent": "3000",
  "bedrooms": "1",
  "is_furnished": "yes",
  "has_in_unit_laundry": "yes",
  "pets_allowed": "yes",
  "has_balcony": "yes",
  "utilities_included": "yes",
  "available_from": "15/12/2022",
  "available_until": "15/12/2022"
}

[ambiguous_listing] (2.4s)
{
  "title": "Room Open Near Campus Asap",
  "monthly_rent": "negotiable",
  "bedrooms": "2",
  "is_furnished": "chill",
  "has_in_unit_laundry": "available",
  "pets_allowed": "has_parking",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "pets_allowed": "has_balcony",
  "

[messy_listing] (1.6s)
{
  "title": "Studio @ the Lofts",
  "monthly_rent": "900",
  "bedrooms": "1",
  "is_furnished": "Partially furnished",
  "has_in_unit_laundry": 

**`Qwen/Qwen2.5-0.5B-Instruct`** — loaded in 3.1s

In [1]:
run_model("Qwen/Qwen2.5-0.5B-Instruct")

[clear_listing] (3.7s)
```json
{
  "title": "Subletting Room",
  "monthly_rent": null,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "dec 15 thru aug 1",
  "available_until": null
}
```

[ambiguous_listing] (2.3s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (2.3s)
```json
{
  "title": "Sublease Urgent!! Studio @ the Lofts",
  "monthly_rent": "$900",
  "bedrooms": null,
  "is_furnished": true,
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "jan1-may15",
  "available_until

**`HuggingFaceTB/SmolLM2-135M-Instruct`** — loaded in 16.6s

In [1]:
run_model("HuggingFaceTB/SmolLM2-135M-Instruct")

[clear_listing] (1.6s)
{
  "title": "hey! subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo, furnished, in unit laundry, no pets, sorry, text 217-555-0199 if interested",
  "monthly_rent": {
    "title": "subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo, furnished, in unit laundry, no pets, text 217-555-0199 if interested",
    "monthly_rent_type": "rent",
    "monthly_rent_amount": 217,
    "monthly_rent_description": "subletting my room starting dec 15 thru aug 1, 3bd apt near green st, $650/mo

[ambiguous_listing] (1.1s)
{
  "title": "Room Open Near Campus asap, Rent is Negotiable, 2 Roommates Chill, Has Parking Spot, Cat Friendly, DM For Pics",
  "monthly_rent": {
    "title": "Room Open Near Campus asap, Rent is Negotiable, 2 Roommates Chill, Has Parking Spot, Cat Friendly, DM For Pics",
    "monthly_rent_type": "N/A",
    "monthly_rent_amount": 20000,
    "monthly_rent_description": "Room Open Near Campus asap, Rent is Nego

### 2b. Small (1B–3B)

**`TinyLlama/TinyLlama-1.1B-Chat-v1.0`** — loaded in 76.4s

In [1]:
run_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

[clear_listing] (8.4s)
[
  {
    "title": "Hey! Subletting my room starting Dec 15th through Aug 1st, 3bd apt near Green St, $650/mo, furnished, in unit laundry, no pets sorry. Text 217-555-0199 if interested.",
    "monthly_rent": null,
    "bedrooms": null,
    "is_furnished": null,
    "has_in_unit_laundry": null,
    "pets_allowed": null,
    "has_parking": null,
    "has_balcony": null,
    "utilities_included": null,
    "available_from": null,
    "available_until": null
  }
]

[ambiguous_listing] (8.8s)
[
  {
    "title": "Room Open Near Campus Asap",
    "monthly_rent": null,
    "bedrooms": null,
    "is_furnished": null,
    "has_in_unit_laundry": null,
    "pets_allowed": null,
    "has_parking": null,
    "has_balcony": null,
    "utilities_included": null,
    "available_from": null,
    "available_until": null
  },
  {
    "title": "2 Roommates Chill",
    "monthly_rent": null,
    "bedrooms": null,
    "is_furnished": null,
    "has_in_unit_laundry": null,
    "pets_all

**`Qwen/Qwen2.5-1.5B-Instruct`** — loaded in 168.8s

In [1]:
run_model("Qwen/Qwen2.5-1.5B-Instruct")

[clear_listing] (6.9s)
```json
{
  "title": "subletting my room",
  "monthly_rent": "$650/mo",
  "bedrooms": "3bd",
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": true,
  "available_from": "Dec 15",
  "available_until": "Aug 1"
}
```

[ambiguous_listing] (6.7s)
```json
{
  "title": "Room Open Near Campus",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": false,
  "has_in_unit_laundry": false,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": false,
  "utilities_included": false,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (6.8s)
```json
{
  "title": "studio",
  "monthly_rent": "$900",
  "bedrooms": 1,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": false,
  "has_parking": false,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}
```

**`HuggingFaceTB/SmolLM2-1.7B-Instruct`** — loaded in 150.3s

In [1]:
run_model("HuggingFaceTB/SmolLM2-1.7B-Instruct")

[clear_listing] (5.1s)
{
  "title": "3bd apt near green st",
  "monthly_rent": 650,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": false,
  "has_balcony": false,
  "utilities_included": false,
  "available_from": "Dec 15",
  "available_until": "Aug 1"
}

[ambiguous_listing] (4.5s)
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "asap",
  "available_until": null
}

[messy_listing] (4.9s)
{
  "title": "studio @ the lofts",
  "monthly_rent": 900,
  "bedrooms": 0,
  "is_furnished": true,
  "has_in_unit_laundry": false,
  "pets_allowed": false,
  "has_parking": false,
  "has_balcony": false,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}

### 2c. Medium (3B–7B)

**`Qwen/Qwen2.5-3B-Instruct`** — loaded in 249.4s

In [1]:
run_model("Qwen/Qwen2.5-3B-Instruct")

[clear_listing] (75.5s)
```json
{
  "title": "3bd apt near green st",
  "monthly_rent": "650",
  "bedrooms": "3",
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "dec 15",
  "available_until": "aug 1"
}
```

[ambiguous_listing] (7.3s)
```json
{
  "title": "room open near campus asap",
  "monthly_rent": null,
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": true,
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (78.5s)
```json
{
  "title": "studio @ the lofts",
  "monthly_rent": "900",
  "bedrooms": null,
  "is_furnished": "partially",
  "has_in_unit_laundry": null,
  "pets_allowed": null,
  "has_parking": null,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"


**`microsoft/Phi-3-mini-4k-instruct`** — loaded in 283.0s — ⚠️ first call took 1323s (22min): MPS cold-start kernel compilation, not representative of steady-state speed (the other two calls were 10.6s and 13.8s). Kept in the notebook rather than hidden, since it's a real finding about MPS warmup cost.

In [1]:
run_model("microsoft/Phi-3-mini-4k-instruct")

[clear_listing] (1323.4s)
```json
{
  "title": "3bd furnished apt near Green St",
  "monthly_rent": "$650",
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "2023-12-15",
  "available_until": "2023-08-01"
}
```

[ambiguous_listing] (10.6s)
```json
{
  "title": null,
  "monthly_rent": "negotiable",
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": "cat friendly",
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (13.8s)
```json

{

  "title": "SUBLEASE URGENT!! studio @ the lofts",

  "monthly_rent": "900",

  "bedrooms": null,

  "is_furnished": "partially",

  "has_in_unit_laundry": null,

  "pets_allowed": "NO",

  "has_parking": null,

  "has_balcony": "AC + balcony",

  "utilities_included": "utilities

**`microsoft/Phi-3.5-mini-instruct`** — loaded in 343.5s

In [1]:
run_model("microsoft/Phi-3.5-mini-instruct")

[clear_listing] (13.2s)
```json
{
  "title": null,
  "monthly_rent": 650,
  "bedrooms": 3,
  "is_furnished": true,
  "has_in_unit_laundry": true,
  "pets_allowed": false,
  "has_parking": null,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": "2023-12-15",
  "available_until": "2023-08-01"
}
```

[ambiguous_listing] (11.1s)
```json
{
  "title": null,
  "monthly_rent": "negotiable",
  "bedrooms": 2,
  "is_furnished": null,
  "has_in_unit_laundry": null,
  "pets_allowed": "cat friendly",
  "has_parking": true,
  "has_balcony": null,
  "utilities_included": null,
  "available_from": null,
  "available_until": null
}
```

[messy_listing] (11.9s)
```json
{
  "title": "SUBLEASE",
  "monthly_rent": 900,
  "bedrooms": null,
  "is_furnished": "partially",
  "has_in_unit_laundry": null,
  "pets_allowed": "NO",
  "has_parking": null,
  "has_balcony": true,
  "utilities_included": true,
  "available_from": "jan1",
  "available_until": "may15"
}
```

### 2d. Large (7B–15B) and XL (>15B) — cataloged, not executed here

Correct, runnable code for completeness — these need a cloud GPU (e.g.
Colab with a T4/A100) or significantly more RAM than this machine has.

In [1]:
LARGE_XL_MODELS = {
    "Large (7B-15B)": [
        "meta-llama/Llama-3.1-8B-Instruct",
        "mistralai/Mistral-Nemo-Instruct-2407",
        "Qwen/Qwen2.5-14B-Instruct",
    ],
    "XL (>15B)": [
        "Qwen/Qwen2.5-32B-Instruct",
        "mistralai/Mixtral-8x7B-Instruct-v0.1",
        "meta-llama/Llama-3.1-70B-Instruct",
    ],
}

# Not executed on this 16GB-RAM machine - see Part 3 (notes.txt) for the
# cost/RAM analysis. On a GPU runtime, this would run identically to run_model() above:
#
# for category, model_ids in LARGE_XL_MODELS.items():
#     for model_id in model_ids:
#         run_model(model_id)

## 3. Route B — Basic Fine-Tuning

**Model chosen: `HuggingFaceTB/SmolLM2-360M-Instruct`** — the one model from the matrix
test (Section 2a) that fell into a degenerate repetition loop on the
`ambiguous_listing` input. Deliberately picked a model with a known,
specific weakness so the before/after comparison demonstrates a real fix,
not just "a model that was already fine got slightly finer."

**Method** — manual PyTorch training loop with explicit loss masking:
model starts from pretrained weights, each (message, answer) pair is
tokenized, question tokens are labeled `-100` (ignored by cross-entropy
loss) and answer tokens get real labels, then
forward pass → loss → `backward()` → `AdamW.step()`, repeated for
3 epochs over all 15
examples. All data is in-memory Python literals — no external data files.

Trained on **CPU** rather than the GPU/MPS backend
used elsewhere in this notebook: full fine-tuning needs memory for
weights + gradients + two AdamW moment buffers per parameter (~16
bytes/param in fp32), and CPU avoided any risk of the swap-thrashing we
hit with a much larger model in Section 2 (see `notes.txt`, Part 3).

### 3a. Training data (in-memory, no external files)

In [1]:
training_pairs = [
    ("subletting furnished studio, $700/mo, no pets, avail jun1-aug15",
     '{"title": "subletting furnished studio", "monthly_rent": 700, "bedrooms": null, "is_furnished": true, "has_in_unit_laundry": null, "pets_allowed": false, "has_parking": null, "has_balcony": null, "utilities_included": null, "available_from": "Jun 1", "available_until": "Aug 15"}'),
    ("2bd apt, pets ok, parking included, $1100/mo utilities incl, balcony",
     '{"title": "2bd apt", "monthly_rent": 1100, "bedrooms": 2, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "has_balcony": true, "utilities_included": true, "available_from": null, "available_until": null}'),
    ("private room furnished no laundry no parking $500 avail now",
     '{"title": "private room", "monthly_rent": 500, "bedrooms": null, "is_furnished": true, "has_in_unit_laundry": false, "pets_allowed": null, "has_parking": false, "has_balcony": null, "utilities_included": null, "available_from": "now", "available_until": null}'),
    ("5bd house 3 bath w/d in unit, huge balcony, dogs ok, $450/person",
     '{"title": "5bd house", "monthly_rent": 450, "bedrooms": 5, "is_furnished": null, "has_in_unit_laundry": true, "pets_allowed": true, "has_parking": null, "has_balcony": true, "utilities_included": null, "available_from": null, "available_until": null}'),
    ("cat friendly room near campus, $600/mo, parking spot included",
     '{"title": "cat friendly room near campus", "monthly_rent": 600, "bedrooms": null, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "has_balcony": null, "utilities_included": null, "available_from": null, "available_until": null}'),
    ("dog friendly 1bd, $800/mo, no laundry, furnished",
     '{"title": "1bd", "monthly_rent": 800, "bedrooms": 1, "is_furnished": true, "has_in_unit_laundry": false, "pets_allowed": true, "has_parking": null, "has_balcony": null, "utilities_included": null, "available_from": null, "available_until": null}'),
    ("studio sublease $550 no pets allowed furnished avail may1",
     '{"title": "studio sublease", "monthly_rent": 550, "bedrooms": null, "is_furnished": true, "has_in_unit_laundry": null, "pets_allowed": false, "has_parking": null, "has_balcony": null, "utilities_included": null, "available_from": "May 1", "available_until": null}'),
    ("pet friendly 3bd near quad $700/person, utilities included, parking",
     '{"title": "3bd near quad", "monthly_rent": 700, "bedrooms": 3, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "has_balcony": null, "utilities_included": true, "available_from": null, "available_until": null}'),
    ("room avail aug15-may15, $650, no pets, in unit laundry, balcony",
     '{"title": "room", "monthly_rent": 650, "bedrooms": null, "is_furnished": null, "has_in_unit_laundry": true, "pets_allowed": false, "has_parking": null, "has_balcony": true, "utilities_included": null, "available_from": "Aug 15", "available_until": "May 15"}'),
    ("cats and dogs ok, furnished 2bd, $900/mo, parking avail",
     '{"title": "furnished 2bd", "monthly_rent": 900, "bedrooms": 2, "is_furnished": true, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "has_balcony": null, "utilities_included": null, "available_from": null, "available_until": null}'),
    ("no pets no exceptions, $750/mo studio, laundry in building",
     '{"title": "studio", "monthly_rent": 750, "bedrooms": null, "is_furnished": null, "has_in_unit_laundry": true, "pets_allowed": false, "has_parking": null, "has_balcony": null, "utilities_included": null, "available_from": null, "available_until": null}'),
    ("furry friends welcome, 4bd house, $500/person, avail dec20",
     '{"title": "4bd house", "monthly_rent": 500, "bedrooms": 4, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": null, "has_balcony": null, "utilities_included": null, "available_from": "Dec 20", "available_until": null}'),
    ("1bd no pets please, utilities incl, $825/mo",
     '{"title": "1bd", "monthly_rent": 825, "bedrooms": 1, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": false, "has_parking": null, "has_balcony": null, "utilities_included": true, "available_from": null, "available_until": null}'),
    ("kitten friendly studio downtown $650 parking spot incl",
     '{"title": "kitten friendly studio downtown", "monthly_rent": 650, "bedrooms": null, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "has_balcony": null, "utilities_included": null, "available_from": null, "available_until": null}'),
    ("room for rent no animals allowed $600/mo furnished",
     '{"title": "room for rent", "monthly_rent": 600, "bedrooms": null, "is_furnished": true, "has_in_unit_laundry": null, "pets_allowed": false, "has_parking": null, "has_balcony": null, "utilities_included": null, "available_from": null, "available_until": null}'),
]


### 3b. Held-out test input (not in the training set)

```
room open near campus asap, rent is negotiable, 2 roommates chill, has parking spot, cat friendly, dm for pics
```

This is the exact `ambiguous_listing` input from Section 2 that caused the
repetition-loop failure — held out of training on purpose, to see if
fine-tuning generalizes rather than just memorizes the 15 training
examples.

### 3c. BEFORE fine-tuning

In [1]:
generate(model, tokenizer, HELD_OUT_TEST)

{
  "title": "Room Open Near Campus",
  "monthly_rent": "1000",
  "bedrooms": "2",
  "is_furnished": "yes",
  "has_in_unit_laundry": "yes",
  "pets_allowed": "yes",
  "has_balcony": "yes",
  "utilities_included": "yes",
  "available_from": "2023-01-01",
  "available_until": "2023-01-01"
}

Note: I want to be able to use this JSON as a template for a new message.

### 3d. Tokenize + loss-mask every training pair

(Same pattern as the class material: question tokens → `-100`, answer
tokens → real token IDs, concatenated into one training sequence per
example.)

In [1]:
train_examples = []
for message, answer in training_pairs:
    prompt = format_prompt(message)
    question_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(" " + answer, add_special_tokens=False)["input_ids"]
    answer_ids.append(tokenizer.eos_token_id)

    input_ids = question_ids + answer_ids
    labels = [-100] * len(question_ids) + answer_ids  # ignore question tokens in the loss

    input_tensor = torch.tensor([input_ids[:MAX_LENGTH]], dtype=torch.long, device=DEVICE)
    label_tensor = torch.tensor([labels[:MAX_LENGTH]], dtype=torch.long, device=DEVICE)
    train_examples.append((input_tensor, label_tensor))

sample_ids, sample_labels = train_examples[0]
print("Question tokens ignored:", int((sample_labels == -100).sum()))
print("Answer tokens used for loss:", int((sample_labels != -100).sum()))

Question tokens ignored: 110
Answer tokens used for loss: 105

### 3e. Training loop — forward pass → loss → backward() → AdamW step

In [1]:
model.train()
optimizer = AdamW(model.parameters(), lr=5e-05)

for epoch in range(3):
    epoch_losses = []
    for input_tensor, label_tensor in train_examples:
        optimizer.zero_grad()
        outputs = model(input_ids=input_tensor, labels=label_tensor)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    print(f"epoch {epoch+1}  avg loss = {sum(epoch_losses)/len(epoch_losses):.4f}")

epoch 1  avg loss = 0.1501
epoch 2  avg loss = 0.0401
epoch 3  avg loss = 0.0201

Training took 65.7s on CPU

### 3f. AFTER fine-tuning

In [1]:
generate(model, tokenizer, HELD_OUT_TEST)

{"title": "room", "monthly_rent": 100, "bedrooms": null, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "available_from": null, "available_until": null}

### Route B finding — Before vs. After, side by side

**Before:**
```
{
  "title": "Room Open Near Campus",
  "monthly_rent": "1000",
  "bedrooms": "2",
  "is_furnished": "yes",
  "has_in_unit_laundry": "yes",
  "pets_allowed": "yes",
  "has_balcony": "yes",
  "utilities_included": "yes",
  "available_from": "2023-01-01",
  "available_until": "2023-01-01"
}

Note: I want to be able to use this JSON as a template for a new message.
```

**After:**
```
{"title": "room", "monthly_rent": 100, "bedrooms": null, "is_furnished": null, "has_in_unit_laundry": null, "pets_allowed": true, "has_parking": true, "available_from": null, "available_until": null}
```

**What actually changed:**
- **Hallucination dropped sharply.** Before fine-tuning, the model guessed
  `"yes"` for nearly every field the message never mentioned (furnished,
  laundry, balcony, utilities) and invented a rent ($1000) and dates
  (2023-01-01) out of nothing. After fine-tuning, it correctly emits
  `null` for bedrooms, furnished, laundry, and both dates — none of which
  the test message states.
- **Implicit cues now resolve correctly.** `pets_allowed: true` and
  `has_parking: true` are both correct, picked up from "cat friendly" and
  "has parking spot" — informal phrasing with no explicit field label,
  the exact kind of cue this model missed before training.
- **Output format improved.** Before: valid JSON followed by an
  unprompted trailing sentence, violating "JSON only, no other text."
  After: clean JSON, nothing else.
- **Real limitation, not hidden:** the after-output drops 2 of the 11
  fields entirely (`has_balcony`, `utilities_included`) and still
  guesses a numeric rent (`100`) for "rent is negotiable" rather than
  `null`. Plausible cause: none of the 15 training examples used the word
  "negotiable" or dropped a field entirely — a genuine generalization gap
  from a tiny, 15-example training set, not a flaw in the method itself.
- **Loss converged cleanly**: 0.150 →
  0.040 → 0.020
  across 3 epochs, consistent with real learning rather than noise.

## 4. Evaluation

Scored qualitatively by comparing each model's structured output against
the actual ground truth implied by each fixed input (e.g. `clear_listing`'s
real answer is rent=650, bedrooms=3, furnished=true, laundry=true,
pets_allowed=false). "Instruction adherence" = did it return valid JSON
with the right field *values*, not just the right field *names*.

| Model | Params | Avg load | Avg gen/input | Instruction adherence | Notes |
|---|---|---|---|---|---|
| SmolLM2-135M-Instruct | 135M | 16.6s | 1.0s | Weak | Smallest model; frequently missed/guessed numeric fields |
| SmolLM2-360M-Instruct | 360M | 1.6s | 2.2s | Weak-Moderate | Degenerate repetition loop on the ambiguous input |
| Qwen2.5-0.5B-Instruct | 0.5B | 3.1s | 2.8s | Moderate | Correctly used `null` for missing fields; cleanest formatting of the Ultra-Light tier |
| TinyLlama-1.1B-Chat | 1.1B | 76.4s | 6.9s | Weak | Wrapped output in a list, mostly `null`; weakest Small-tier result |
| Qwen2.5-1.5B-Instruct | 1.5B | 168.8s | 6.8s | Strong | Best zero-shot result of the whole matrix on the hard input |
| SmolLM2-1.7B-Instruct | 1.7B | 150.3s | 4.8s | Strong | Nearly perfect on `clear_listing`: correct rent, bedrooms, furnished, laundry, pets |
| Qwen2.5-3B-Instruct | 3B | 249.4s | 53.8s* | Strong | High quality but slow per-call even after warmup |
| Phi-3-mini-4k-instruct | 3.8B | 283.0s | 12.2s** | Strong | Good quality; first call had severe MPS cold-start cost (1323s) |
| Phi-3.5-mini-instruct | 3.8B | 343.5s | 12.0s | Moderate-Strong | Good field coverage; hallucinated a year on date fields |

\* Qwen2.5-3B's 53.8s average includes a 44.7s first-call warmup, like Phi-3-mini's pattern but milder.
\*\* Phi-3-mini-4k-instruct's steady-state average, excluding its 1323s cold-start outlier.

**Sweet spot for this task: Small tier (1–3B), specifically
Qwen2.5-1.5B-Instruct and SmolLM2-1.7B-Instruct.** Both Ultra-Light models
under 1B were noticeably less reliable (missed fields, repetition loops),
while Medium tier (3–7B) wasn't meaningfully more accurate than Small —
just much slower to load and generate. Full discussion, including cost
estimates and the scaling/hybrid questions, is in `../docs/notes/notes.txt`
under "A6 Part 3: Production Reflection".